## Mise en place

**Dans Colab, exécutez la cellule suivante une fois**, avant toutes les autres : elle dépose le module `mcdp_utils.py` et le dossier `data` du cours à côté du notebook.
Sur votre machine, où ces fichiers sont déjà en place, elle ne télécharge rien et ne change rien.

In [ ]:
"""Mise en place : le module et les donnees du cours, a cote du notebook."""
# Dans Colab, la machine pretee est vide : cette cellule telecharge depuis le
# depot du cours ce qui manque, et rien d'autre. Aucune installation.
import os
import urllib.request

RAW_BASE = "https://raw.githubusercontent.com/adilion1/cours-monte-carlo/main/notebooks/"
FICHIERS = ("mcdp_utils.py", "data/prices.csv", "data/returns_daily.csv")

os.makedirs("data", exist_ok=True)
_manquants = [_nom for _nom in FICHIERS if not os.path.exists(_nom)]
for _nom in _manquants:
    print("telechargement :", _nom)
    urllib.request.urlretrieve(RAW_BASE + _nom, _nom)

print("Prêt : module et données en place.")

# TP du chapitre 0 — de la première cellule aux 4 151 séances

**Master 2 Finance — Optimisation dynamique et simulations de Monte Carlo en Python**
Chapitre 0, *Outils : Python de zéro, sans rien installer*. Support : les unités `u01` à `u07`.

---

## Ce que ce TP vous demande

Vous n'avez peut-être jamais écrit une ligne de code il y a deux heures. À la fin de ce notebook,
vous aurez produit une figure de seize ans de marché, réparé quatre messages d'erreur, extrait le
dernier cours de six actifs, sorti la pire année d'un portefeuille, écrit deux fonctions prouvées
par un test, et **mesuré** ce que la vectorisation fait gagner sur un million de valeurs.

Sept parties, **quatorze fonctions courtes** à écrire — onze fonctions de calcul, d'une à quatre
lignes chacune, plus trois cellules d'écrit (les deux dictionnaires de prédiction et le commentaire
final) —, et à chaque fois un chiffre du fil rouge à retrouver : 4 151 séances du 5 janvier 2010 au 4 septembre 2026, et un portefeuille de six lignes
(S&P 500 40 %, CAC 40 20 %, AAPL 10 %, LVMH 10 %, GLD 10 %, TLT 10 %).

**TP non noté.** Les trois TP notés du cours sont ceux des chapitres 4, 7 et 11. Ici, la note est
remplacée par les `verifier(...)` : chaque partie se corrige toute seule, et chaque échec vous dit
ce qui était attendu et où regarder.

**Aucune probabilité, et une seule notion de finance.** La partie E définit, au sens strictement
minimal, ce que rapporte à l'échéance un droit d'acheter à un prix fixé : c'est un `max` à
programmer, rien d'autre — l'objet financier complet est au **ch01 u05**. Deux tirages au sort
apparaissent en partie F, uniquement pour montrer qu'une même graine redonne les mêmes nombres.
Rien n'est simulé ici : la simulation, et la recette en quatre étapes qui l'encadre, commencent au
chapitre 2.

## Objectifs (vérifiables ici même)

| # | Vous saurez | Où c'est vérifié |
|---|---|---|
| **O-A** | exécuter une cellule, produire une figure à partir des données du cours, en changer l'actif | partie A |
| **O-B** | provoquer, lire et réparer les quatre messages d'erreur qui bloquent le plus, et constater qu'aucun n'abîme rien | partie B |
| **O-C** | ranger des prix dans une liste et dans un dictionnaire, en extraire une tranche sans abîmer l'original | partie C |
| **O-D** | écrire un accumulateur correct (initialisation sur une valeur observée, mise à jour **dans** le `if`) | partie D |
| **O-E** | écrire une fonction documentée et la mettre à l'épreuve sur un cas dont vous connaissez la réponse | partie E |
| **O-F** | remplacer une boucle par une opération de tableau, **mesurer** le gain, et éviter les deux fautes muettes (vue/copie, `ddof`) | partie F |

## Temps

| Partie | Contenu | Durée |
|---|---|---|
| **0** | Trois prédictions écrites (obligatoire, non noté) | 3 min |
| **A** | La courbe (u01) | 5 min |
| **B** | Réparer quatre erreurs (u02) | 10 min |
| **C** | Ranger des prix (u03) | 10 min |
| **D** | La pire année (u04) | 15 min |
| **E** | Nommer un calcul, le mettre à l'épreuve (u05) | 15 min |
| **F** | Passer à l'échelle (u06) | 20 min |
| **G** | Confrontation des prédictions | 5 min |
| | **Total mains sur clavier** | **83 min** |
| **H** | Passer en local (u07) — facultatif, **obligatoire avant le ch05** | hors TP |

En séance, les parties A et B sont faites ensemble (25 min) ; C à G se terminent à la maison.

## Trois conventions, à lire une fois

1. **Le notebook se lit et s'exécute strictement dans l'ordre**, du haut vers le bas. La cellule 1
   charge les données et définit les constantes ; toutes les autres en dépendent. Si vous obtenez
   un `NameError` sur un nom que vous voyez à l'écran, c'est l'ordre d'exécution qui est en cause :
   *Kernel → Restart & Run All* (u04).
2. **Cellules à trous.** Vous **remplacez** le bloc `# TODO: … / raise NotImplementedError("TODO")`
   par votre code et **vous gardez tout ce qui suit** : le `return`, les vérifications et les
   affichages font partie de l'exercice. Tant que le `raise` est là, les lignes en dessous sont
   inatteignables — c'est normal, ce n'est pas une panne.
3. **Aide en trois temps.** Après chaque partie, une cellule repliée contient `indice(...)` — une
   étape du raisonnement — puis `solution(...)` — la réponse, **toutes deux écrites en commentaire**.
   Retirez le `#` de la ligne voulue quand vous en avez besoin, pas avant : le `verifier(...)` qui
   précède vous dit déjà où regarder. C'est aussi pourquoi un *Restart & Run All* ne vous montre
   jamais les réponses.
4. **Deux décorations que vous n'avez pas à écrire.** Les fonctions de ce TP sont données sous la
   forme `def rendement(prix_debut: float, prix_fin: float) -> float:` : les `: float` et la flèche
   `-> float` disent, **pour un lecteur humain**, que les entrées et la sortie sont des nombres à
   virgule. Python ne les vérifie pas. De même, la docstring est développée en sections
   `Parameters` / `Returns` : c'est la convention de la bibliothèque numpy, adoptée par les TP du
   cours. Les deux sont facultatives — la forme d'u05, `def rendement(prix_debut, prix_fin):` avec
   une docstring d'une phrase, est correcte. Ne les supprimez pas ici : elles vous disent l'ordre
   et la nature de ce qu'il faut rendre.

## Les quinze primitives numpy du cours

Jusqu'au chapitre 5, vous n'écrirez rien d'autre que celles-ci :

`np.array` · `.shape` · indexation et tranche · masque booléen · `np.maximum` · `np.mean` ·
`np.std` · `np.percentile` · `np.exp` / `np.log` · `np.cumsum` · `np.sqrt` ·
`np.random.default_rng(SEED)` · `rng.standard_normal` · `rng.choice` · `rng.spawn` ;
plus `plt.subplots` et `ax.plot` / `ax.hist`.

Tout le reste — à commencer par `charger_fil_rouge()` et `payoff_call()` — s'**appelle**, ne se
réécrit pas. Deux règles de calcul viennent avec (`NOTATION.md` §12) : écart-type toujours en
`ddof=1`, et **un** `SEED`, **un** `rng`, toute répétition par `rng.spawn(k)` — jamais `SEED + i`.

In [ ]:
"""Chapter 0 lab -- setup cell: imports, red-thread data, seed, sanity checks."""
%matplotlib inline

import os
import sys

# Find mcdp_utils.py whether this notebook runs from cours_v2/ch00/ (local, after u07)
# or from /content (Colab, after the three files have been uploaded -- chapitre.md).
for _candidat in (".", "..", "../..", "/content"):
    if os.path.exists(os.path.join(_candidat, "mcdp_utils.py")):
        sys.path.insert(0, _candidat)
        break
else:
    raise FileNotFoundError(
        "mcdp_utils.py est introuvable. En Colab : televersez mcdp_utils.py et un dossier "
        "data/ contenant prices.csv et returns_daily.csv (voir chapitre.md, section "
        "'Zero installation'). En local : lancez ce notebook depuis cours_v2/ch00/."
    )

import time

import matplotlib.pyplot as plt
import numpy as np

from mcdp_utils import charger_fil_rouge, indice, payoff_call, set_style, solution, verifier

set_style()

# One SEED, one rng, created once (NOTATION.md §12).  This chapter only uses them to
# show that a fixed seed gives back the very same numbers; nothing here is simulated.
SEED = 42
rng = np.random.default_rng(SEED)

FIL = charger_fil_rouge()                # the single entry point to the data
DATES = FIL["dates"]                     # 4151 trading days, datetime64[D]
PRIX = FIL["prix"]                       # asset -> ndarray of closing prices
RLOG = FIL["rendements_log"]             # asset -> ndarray of daily log returns
LIGNES = ["SP500", "CAC40", "AAPL", "LVMH", "GLD", "TLT"]   # the six portfolio lines
K = FIL["S0"]                            # 7718.60, S&P 500 close on 2026-09-04

# The seventeen calendar-year performances of the red-thread portfolio and of the S&P 500
# alone, rounded to four decimals (ch00 u04; recomputed by figures_1.py from the same data).
ANNEES = [2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018,
          2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]
PERFS_PTF = [0.1771, 0.0307, 0.1505, 0.1241, 0.1140, 0.0298, 0.0973, 0.2036, -0.0411,
             0.3468, 0.2116, 0.2347, -0.1528, 0.2079, 0.1248, 0.1740, 0.0374]
PERFS_SP500 = [0.1102, -0.0002, 0.1152, 0.3180, 0.1139, 0.0022, 0.0850, 0.1942, -0.0624,
               0.2888, 0.1626, 0.2689, -0.1944, 0.2423, 0.2331, 0.1639, 0.1275]


def rappel(nom: str, gabarit: str = "{}", defaut: str = "(a completer)") -> str:
    """Format a value bound by an earlier cell, or a placeholder if it is missing.

    Used only by the closing cell, so that the statement notebook can be run top to
    bottom before every TODO has been filled in.
    """
    valeur = globals().get(nom, None)
    return defaut if valeur is None else gabarit.format(valeur)


print(f"python {sys.version.split()[0]} | numpy {np.__version__}")
print(f"{DATES.size} seances, du {DATES[0]} au {DATES[-1]} | {len(PRIX)} actifs disponibles")
print(f"S&P 500 : {PRIX['SP500'][0]:,.2f} -> {PRIX['SP500'][-1]:,.2f} "
      f"(x{PRIX['SP500'][-1] / PRIX['SP500'][0]:.2f})")

assert DATES.size == 4151, "the red-thread panel should hold 4151 aligned sessions"
assert abs(K - 7718.60) < 1e-3, "S0 moved: check data/stats_summary.md"
assert len(ANNEES) == len(PERFS_PTF) == len(PERFS_SP500) == 17, "seventeen calendar years"

---

## Partie 0 — Trois prédictions écrites (3 min, obligatoire, non noté)

**Ne lancez aucune autre cellule avant d'avoir rempli celle-ci.** Une prédiction écrite avant le
calcul ne sert pas à avoir raison : elle sert à rendre visible l'écart entre ce que vous croyiez et
ce que les données disent. Sans elle, on lit le bon chiffre et on se persuade qu'on l'attendait.

Trois questions, trois nombres, aucun calcul :

1. **`minutes_jusqu_a_la_figure`** — en partant de ce notebook, combien de **minutes** vous
   semblent nécessaires pour voir apparaître la courbe des seize ans du S&P 500 ?
2. **`pire_annee_pct`** — le portefeuille du cours a traversé dix-sept années civiles depuis 2010.
   Quelle a été la **pire performance annuelle**, en pour-cent (un nombre négatif) ?
3. **`seances_sous_moins_2pct`** — sur les 4 151 séances, combien ont perdu **plus de 2 % en une
   seule journée** ?

Écrivez vos trois nombres. La partie G les relira à côté des trois chiffres mesurés.

In [ ]:
def mes_predictions() -> dict:
    """The three numbers written down BEFORE running anything else.

    Returns
    -------
    dict
        Keys ``minutes_jusqu_a_la_figure`` (minutes), ``pire_annee_pct`` (worst
        calendar-year performance of the red-thread portfolio, in percent, negative)
        and ``seances_sous_moins_2pct`` (number of sessions below -2 % in one day).
    """
    # TODO: Renvoyer vos trois predictions dans un dictionnaire portant exactement
    #      les trois cles de la docstring ; n'importe quel nombre honnete convient,
    #      rien n'est note ici.
    raise NotImplementedError("TODO")


PREDICTIONS = mes_predictions()

assert set(PREDICTIONS) == {
    "minutes_jusqu_a_la_figure",
    "pire_annee_pct",
    "seances_sous_moins_2pct",
}, "the dict must carry exactly the three keys of the docstring"
assert PREDICTIONS["minutes_jusqu_a_la_figure"] > 0.0, (
    "minutes_jusqu_a_la_figure attend un nombre de minutes strictement positif"
)
assert PREDICTIONS["pire_annee_pct"] < 0.0, "a worst year is a loss: write a negative number"
assert 0 <= PREDICTIONS["seances_sous_moins_2pct"] <= 4151, (
    "seances_sous_moins_2pct attend un entier entre 0 et 4151 (le panel compte 4151 seances)"
)
for cle, valeur in PREDICTIONS.items():
    print(f"{cle:>28s} : {valeur}")

---

## Partie A — Six lignes suffisent pour voir seize ans de marché (5 min, u01)

**Le problème.** Le S&P 500 valait 1 136,52 le 5 janvier 2010 et 7 718,60 le 4 septembre 2026. Ces
deux nombres ne disent rien du chemin : ni le décrochage de mars 2020, ni le creux de 2022, ni la
raideur de 2024-2026. Un tableau de 4 151 lignes ne répond pas non plus — personne ne lit 4 151
nombres. Une courbe répond en une seconde.

Vous écrivez une fonction qui prend le nom d'un actif et trace ses clôtures. Deux gestes, et deux
seulement : `plt.subplots()` **fabrique** la feuille et sa zone dessinable, `ax.plot` y **dessine**.

La fonction rend `(fig, ax)` — la feuille et sa zone — pour que les vérifications puissent regarder
ce qui a réellement été tracé. C'est la première utilité d'un `return` : rendre un objet à celui
qui a appelé, plutôt que de l'afficher et de le perdre.

In [ ]:
def tracer_actif(fil: dict, actif: str) -> tuple:
    """Draw the closing prices of one asset of the red-thread panel.

    Parameters
    ----------
    fil : dict
        What ``charger_fil_rouge()`` returns; ``fil["dates"]`` holds the dates and
        ``fil["prix"][actif]`` the closing prices of one asset.
    actif : str
        Ticker of the asset to draw, e.g. ``"SP500"`` or ``"GLD"``.

    Returns
    -------
    (fig, ax)
        The matplotlib figure and its axes, so that the checks below can inspect what
        has been drawn.
    """
    fig, ax = plt.subplots()
    # TODO: Sur ax, tracer fil["prix"][actif] en fonction de fil["dates"] avec
    #      lw=0.9, puis poser le nom de l'axe des x, celui de l'axe des y, et un
    #      titre qui contient le code de l'actif.
    raise NotImplementedError("TODO")
    return fig, ax

In [ ]:
"""Draw the S&P 500, then the same line of code on another asset."""
fig, ax = tracer_actif(FIL, "SP500")

verifier(
    PRIX["SP500"].size == 4151,
    f"OK : {PRIX['SP500'].size} clotures du S&P 500 sont chargees, "
    f"de {DATES[0]} a {DATES[-1]}.",
    "attendu 4151 clotures : le panel du cours compte 4151 seances alignees. "
    "Si vous en avez 4152, vous lisez prices.csv a la main au lieu d'appeler "
    "charger_fil_rouge() (u01).",
)
verifier(
    len(ax.lines) == 1 and ax.lines[0].get_xydata().shape[0] == 4151,
    f"OK : une seule courbe est tracee, et elle porte "
    f"{ax.lines[0].get_xydata().shape[0] if ax.lines else 0} points.",
    "attendu une courbe de 4151 points sur ax : verifiez que vous appelez ax.plot une "
    "fois, avec fil['dates'] en abscisse et fil['prix'][actif] en ordonnee.",
)
_x = np.asarray(ax.lines[0].get_xdata()) if ax.lines else np.array([])
verifier(
    _x.size == 4151 and _x.dtype.kind in "MO" and str(_x[0])[:10] == "2010-01-05",
    f"OK : l'abscisse porte bien des dates, de {DATES[0]} a {DATES[-1]} -- et non les "
    "positions 0, 1, 2 ... des points.",
    "l'axe des x n'affiche pas des dates mais des numeros de points (0, 1, 2 ...). "
    "C'est ce qui arrive quand on ecrit ax.plot(fil['prix'][actif]) avec un seul "
    "argument : il faut ax.plot(fil['dates'], fil['prix'][actif]). La figure est "
    "lisible, elle est fausse -- seize ans de marche ne sont pas a l'ecran.",
)
verifier(
    ax.get_xlabel() != "" and ax.get_ylabel() != "" and "SP500" in ax.get_title(),
    f"OK : les deux axes sont nommes et le titre porte le code de l'actif "
    f"({ax.get_title()!r}).",
    f"attendu un xlabel, un ylabel et un titre contenant le code de l'actif ; obtenu "
    f"xlabel={ax.get_xlabel()!r}, ylabel={ax.get_ylabel()!r}, titre={ax.get_title()!r}. "
    "Une figure sans axes nommes ne se rend pas.",
)
plt.show()

fig_or, ax_or = tracer_actif(FIL, "GLD")
verifier(
    abs(PRIX["GLD"][-1] - 406.77) < 0.01,
    f"OK : la meme ligne de code trace un autre actif ; GLD termine a "
    f"{PRIX['GLD'][-1]:.2f}, le S&P 500 a {PRIX['SP500'][-1]:.2f}.",
    "attendu 406,77 pour la derniere cloture de GLD : la cle est 'GLD', en majuscules.",
)
plt.show()

In [ ]:
"""Aide de la partie -- retirez le # d'une ligne APRES avoir essaye."""
# indice(...) donne une etape du raisonnement ; solution(...) donne la reponse.
# indice("Deux lignes suffisent pour le trace : ax.plot(fil['dates'], fil['prix'][actif], lw=0.9). "
#        "Les crochets s'enchainent parce que fil est un dictionnaire dont la valeur rangee sous "
#        "'prix' est elle-meme un dictionnaire (u03).")
# solution("ax.plot(fil['dates'], fil['prix'][actif], lw=0.9) puis ax.set_xlabel(...), "
#          "ax.set_ylabel(...), ax.set_title(f'{actif} : ...'). plt.subplots() fabrique, "
#          "ax.plot dessine : c'est toute la repartition des roles.")

---

## Partie B — Quatre erreurs provoquées, puis le premier calcul du cours (10 min, u02)

**Le problème.** Le S&P 500 a clôturé à 7 747,71 le 3 septembre 2026 et à 7 718,60 le lendemain.
« 29,11 points de baisse » ne se compare à rien : le CAC 40 clôture autour de 8 278 et l'action
AAPL autour de 320. Le seul chiffre comparable est la variation **en pourcentage** —
$R = S_1/S_0 - 1$, le rendement simple d'u02.

Avant de l'écrire, on provoque exprès les quatre messages d'erreur qui bloquent le plus. La cellule
ci-dessous les déclenche **dans un filet** (`try` / `except`) : elle affiche le nom et la phrase de
chaque message, et vérifie ensuite qu'une valeur rangée **avant** les quatre pannes est toujours
là. C'est la démonstration numérique de l'idée reçue la plus coûteuse du chapitre : *une erreur
Python n'abîme rien*.

In [ ]:
"""Provoke the four beginner errors on purpose, then show that nothing was damaged."""
temoin = 7718.600098          # bound BEFORE the four failures, re-read after them

CASSE = {
    "SyntaxError": 'print(f"cloture : {7718.60:.2f}"',
    "NameError": "variation = aujourdui / hier - 1.0",
    "TypeError": '"7718.60" - 7747.71',
    "IndentationError": "hier = 7747.71\n    aujourdhui = 7718.60",
}

CASSE_ATTENDU = {}
for nom in CASSE:
    CASSE_ATTENDU[nom] = nom

attrapes = {}
for attendu, fragment in CASSE.items():
    try:
        exec(fragment, {"hier": 7747.71})
    except Exception as erreur:
        attrapes[attendu] = type(erreur).__name__
        print(f"{attendu:>17s} -> {type(erreur).__name__}: {erreur}")

verifier(
    attrapes == CASSE_ATTENDU,
    "OK : les quatre messages sont ceux annonces, chacun nomme son coupable "
    "(la parenthese, le nom mal orthographie, les deux types, l'indentation).",
    f"attendu les quatre memes noms de message ; obtenu {attrapes}. Si l'un manque, "
    "c'est que le fragment correspondant s'est execute sans erreur.",
)
verifier(
    temoin == 7718.600098 and PRIX["SP500"].size == 4151,
    f"OK : apres quatre pannes, temoin vaut toujours {temoin} et les 4 151 clotures "
    "sont intactes. Une cellule qui echoue s'interrompt, elle n'efface rien.",
    "temoin ou les donnees ont bouge : ce ne peut pas venir des quatre erreurs.",
)

Maintenant la ligne qui compte. Écrivez la fonction qui rend le rendement simple de la **dernière**
séance du panel, à partir des deux derniers prix. Deux indices de position suffisent : `-1` est le
dernier élément, `-2` l'avant-dernier (u03 les nommera).

Une précision d'ingénieur, pour que le chiffre affiché ne vous surprenne pas : en tapant
`7718.60 / 7747.71 - 1` à la main on obtient `-0.003757239…`, alors qu'en repartant du fichier on
obtient `-0.003757222…`. L'écart, $1{,}8 \times 10^{-8}$, ne vient pas d'une donnée plus fine : le
fichier range les clôtures en **simple précision** (environ sept chiffres significatifs), si bien
que 7 747,71 y est écrit `7747.709961`. Le nombre exact est celui que vous tapez ; c'est le fichier
qui l'approche. Les deux s'affichent `-0.3757 %`. La vérification ci-dessous porte sur le
**second**, celui que produisent les données du cours.

In [ ]:
def rendement_derniere_seance(fil: dict) -> float:
    """Return the simple return of the S&P 500 on the last session of the panel.

    Parameters
    ----------
    fil : dict
        What ``charger_fil_rouge()`` returns.

    Returns
    -------
    float
        ``S_1 / S_0 - 1`` where ``S_0`` and ``S_1`` are the last two closing prices of
        the S&P 500, in decimal (not in percent).
    """
    prix = fil["prix"]["SP500"]
    # TODO: Renvoyer le rendement simple entre les deux derniers prix, lus avec les
    #      indices -2 (le depart) et -1 (l'arrivee).
    raise NotImplementedError("TODO")

In [ ]:
"""Check the first computation of the course, and display it the readable way."""
RENDEMENT = rendement_derniere_seance(FIL)
print(f"rendement du {DATES[-1]} : {RENDEMENT:.4%}")

verifier(
    abs(RENDEMENT + 0.0037572215721202) < 1e-12,
    f"OK : {RENDEMENT:.4%} le {DATES[-1]}, soit une baisse de "
    f"{PRIX['SP500'][-2] - PRIX['SP500'][-1]:.2f} points sur {PRIX['SP500'][-2]:,.2f}.",
    "attendu -0,0037572215721202 : si vous obtenez +0,0037714..., vous avez divise le "
    "prix de depart par le prix d'arrivee (relire la formule R = S_1 / S_0 - 1) ; si vous "
    "obtenez -29,11, vous avez oublie le rapport et garde la difference.",
)

In [ ]:
"""Aide de la partie -- retirez le # d'une ligne APRES avoir essaye."""
# indice(...) donne une etape du raisonnement ; solution(...) donne la reponse.
# indice("Le prix de depart est l'avant-dernier, le prix d'arrivee le dernier. En Python, "
#        "prix[-1] est le dernier element et prix[-2] l'avant-dernier : aucun besoin de savoir "
#        "combien il y en a.")
# solution("return prix[-1] / prix[-2] - 1.0 -- et l'affichage lisible est "
#          "print(f'rendement : {RENDEMENT:.4%}'), ou .4% signifie 'en pourcentage, quatre decimales'.")

---

## Partie C — Une liste range dans l'ordre, un dictionnaire range par nom (10 min, u03)

**Le problème.** Deux besoins qui reviennent tous les jours sur un desk. D'abord regarder une
tendance courte : les **trois derniers** cours, sans relire les 4 151 autres. Ensuite manipuler six
actifs à la fois en écrivant « le dernier cours d'AAPL », et non « le quatrième élément » — parce
que le jour où l'ordre des lignes change, le quatrième élément change de sens sans prévenir.

Deux fonctions, deux structures :

| Fonction | Rend | Structure |
|---|---|---|
| `trois_derniers(prix)` | les trois dernières valeurs, dans une **nouvelle** liste | une tranche `[-3:]` |
| `derniers_cours(fil, lignes)` | `{"SP500": 7718.60, "CAC40": …}` | un dictionnaire, une clé par actif |

Une exigence sur la première : elle ne doit **pas** modifier ce qu'on lui passe. C'est la règle des
fonctions pures (u05), et la vérification la teste explicitement.

In [ ]:
def trois_derniers(prix) -> list:
    """Return the last three prices of a series, in a NEW list.

    Parameters
    ----------
    prix : list or ndarray
        Prices in chronological order; at least three of them.

    Returns
    -------
    list
        The last three prices, oldest first.  The argument is left untouched: the
        caller's series must never be modified by this function.
    """
    # TODO: Renvoyer list(prix[-3:]) : la tranche prend les trois derniers elements,
    #      et list() garantit une nouvelle liste meme si l'argument est un tableau
    #      numpy.
    raise NotImplementedError("TODO")


def derniers_cours(fil: dict, lignes: list) -> dict:
    """Return the last closing price of each asset, keyed by ticker.

    Parameters
    ----------
    fil : dict
        What ``charger_fil_rouge()`` returns.
    lignes : list of str
        Tickers to read, e.g. ``["SP500", "CAC40", ...]``.

    Returns
    -------
    dict
        ``{ticker: last closing price}``, one entry per ticker of ``lignes``.
    """
    resultat = {}
    # TODO: Parcourir les codes d'actif de `lignes` avec une boucle for et, pour
    #      chacun, ranger float(fil["prix"][actif][-1]) sous ce code dans
    #      `resultat`.
    raise NotImplementedError("TODO")
    return resultat

In [ ]:
"""Check both functions, then check that the first one does not damage its argument."""
verifier(
    trois_derniers([1, 2, 3, 4, 5]) == [3, 4, 5],
    f"OK : sur [1, 2, 3, 4, 5], trois_derniers rend {trois_derniers([1, 2, 3, 4, 5])}.",
    "attendu [3, 4, 5]. Si vous obtenez [1, 2, 3], la tranche est prix[:3] au lieu de "
    "prix[-3:] ; si vous obtenez [4, 5], la borne est mal placee (u03).",
)

essai = [1.0, 2.0, 3.0, 4.0, 5.0]
extrait = trois_derniers(essai)
extrait[0] = 0.0
verifier(
    essai == [1.0, 2.0, 3.0, 4.0, 5.0],
    "OK : ecrire dans le resultat n'a pas touche a la liste de l'appelant.",
    f"la liste de l'appelant a ete abimee : elle vaut {essai}. Vous avez rendu un alias "
    "au lieu d'une nouvelle liste (u03, le piege 'copie = prix').",
)

DERNIERS = derniers_cours(FIL, LIGNES)
for actif, cours in DERNIERS.items():
    print(f"{actif:>6s} : {cours:>10,.2f}")

verifier(
    abs(DERNIERS["GLD"] - 406.77) < 0.01 and abs(DERNIERS["SP500"] - 7718.60) < 0.01,
    f"OK : six cours nommes, dont GLD a {DERNIERS['GLD']:.2f} et le S&P 500 a "
    f"{DERNIERS['SP500']:,.2f}.",
    "attendu 406,77 pour GLD et 7 718,60 pour SP500 : l'indice du dernier element est -1, "
    "et la cle est le code de cotation en majuscules ('GLD', pas 'Gold').",
)
verifier(
    sorted(DERNIERS) == sorted(LIGNES),
    "OK : une cle par ligne du portefeuille, aucune de plus.",
    f"attendu les six cles {sorted(LIGNES)} ; obtenu {sorted(DERNIERS)}.",
)

Le piège de l'unité, en quatre lignes : `copie = prix` ne copie rien. La cellule ci-dessous le
provoque, puis le répare. Aucun message d'erreur n'apparaît — c'est précisément ce qui le rend
dangereux ; le symptôme, en pratique, est un résultat qui change quand on réexécute deux fois la
même cellule.

In [ ]:
"""The alias trap of u03: two names, one single list."""
prix = [7711.76, 7686.14, 7631.47, 7666.60, 7747.71, 7718.60]
copie = prix                     # no copy at all: a second name on the same list
copie[0] = 0.0
avec_alias = prix[0]

prix = [7711.76, 7686.14, 7631.47, 7666.60, 7747.71, 7718.60]
copie = prix.copy()              # an actual, independent copy
copie[0] = 0.0
avec_copie = prix[0]

print(f"apres 'copie = prix'        : prix[0] = {avec_alias}")
print(f"apres 'copie = prix.copy()' : prix[0] = {avec_copie}")

verifier(
    avec_alias == 0.0 and avec_copie == 7711.76,
    "OK : l'alias a bien contamine prix (0.0), la copie ne l'a pas fait (7 711,76). "
    "Deux noms sur un seul objet, contre deux objets independants.",
    f"attendu 0.0 puis 7711.76 ; obtenu {avec_alias} puis {avec_copie}.",
)

In [ ]:
"""Aide de la partie -- retirez le # d'une ligne APRES avoir essaye."""
# indice(...) donne une etape du raisonnement ; solution(...) donne la reponse.
# indice("Une tranche de liste rend deja une nouvelle liste ; list(...) autour ne coute rien et "
#        "protege le cas ou l'appelant passe un tableau numpy, dont la tranche, elle, est une vue "
#        "(vous le verrez en partie F).")
# solution("return list(prix[-3:]) ; et pour le dictionnaire, une boucle for actif in lignes: "
#          "resultat[actif] = float(fil['prix'][actif][-1]).")

---

## Partie D — La pire année sort en dix lignes (15 min, u04)

**Le problème.** Un client vous demande ce qu'il aurait pu perdre en tenant ce portefeuille depuis
2010. Dix-sept années civiles, dix-sept performances : les comparer à l'œil demande dix-sept
lectures et une mémoire de ce qu'on a vu passer. C'est le travail qu'on délègue.

Vous écrivez `pire_annee(annees, perfs)`, qui rend un couple `(année, performance)`. Le motif
s'appelle un **accumulateur** : une variable mise à jour à chaque tour pour retenir un résultat
partiel. Trois décisions, et la fonction est juste :

1. **partir d'une valeur observée** — `perfs[0]`, jamais `0` ;
2. **comparer** chaque performance à celle qu'on a retenue ;
3. **remplacer les deux noms ensemble**, à l'intérieur du `if` : la performance et son année.

La troisième vérification ci-dessous, `pire_annee([2020, 2021], [0.05, 0.02])`, est là pour
attraper la première : sur deux années qui montent, une initialisation à `0` rend une année qui
n'existe pas.

In [ ]:
def pire_annee(annees: list, perfs: list) -> tuple:
    """Return the worst year of a series and its performance.

    Parameters
    ----------
    annees : list of int
        Calendar years, in the same order as ``perfs``.
    perfs : list of float
        Performance of each year, in decimal (``-0.1528`` for -15,28 %).

    Returns
    -------
    (int, float)
        The year holding the smallest performance, and that performance.  On ties the
        first one encountered wins.
    """
    # TODO: Initialiser l'accumulateur sur la PREMIERE paire observee (jamais sur
    #      0), parcourir les deux listes en parallele avec zip(annees, perfs), et
    #      mettre a jour DANS le if a la fois la performance retenue et son annee ;
    #      renvoyer ensuite la paire.
    raise NotImplementedError("TODO")

In [ ]:
"""Check the accumulator on the red thread, on the S&P 500 alone, and on a rising series."""
print(f"couple rendu, portefeuille : {pire_annee(ANNEES, PERFS_PTF)}")
print(f"couple rendu, S&P 500 seul : {pire_annee(ANNEES, PERFS_SP500)}")
annee_ptf, perf_ptf = pire_annee(ANNEES, PERFS_PTF)
annee_sp, perf_sp = pire_annee(ANNEES, PERFS_SP500)

verifier(
    annee_ptf == 2022 and abs(perf_ptf + 0.1528) < 1e-3,
    f"OK : la pire annee du portefeuille est {annee_ptf}, a {perf_ptf:+.2%}.",
    f"attendu le couple (2022, -0.1528) ; obtenu {pire_annee(ANNEES, PERFS_PTF)}. "
    "Trois causes possibles, dans l'ordre ou il faut les regarder. (1) Si le couple "
    "affiche commence par une performance et finit par une annee, votre return rend les "
    "deux valeurs dans l'ordre inverse : la docstring demande (annee, performance) -- "
    "ecrivez return annee_pire, pire. (2) Si vous lisez +3,74 %, la mise a jour est sortie "
    "du if et vous gardez la derniere annee. (3) Si l'annee ne colle pas a la performance, "
    "les deux affectations ne sont pas ensemble dans le if (u04).",
)
print(f"portefeuille : {annee_ptf} a {perf_ptf:+.2%}")
print(f"S&P 500 seul : {annee_sp} a {perf_sp:+.2%}")
verifier(
    annee_sp == 2022 and abs(perf_sp + 0.1944) < 1e-3,
    f"OK : la meme fonction, appelee sur une autre serie, donne {annee_sp} a "
    f"{perf_sp:+.2%} pour le S&P 500 seul -- soit {100 * (perf_ptf - perf_sp):.2f} points "
    "de moins de perte pour le melange de six lignes, la meme annee.",
    "attendu (2022, -0,1944) sur PERFS_SP500.",
)
verifier(
    pire_annee([2020, 2021], [0.05, 0.02]) == (2021, 0.02),
    "OK : sur deux annees qui montent, la pire est la moins bonne des deux -- +2,00 %, "
    "et non 0. L'accumulateur part bien d'une valeur observee.",
    f"attendu (2021, 0.02) ; obtenu {pire_annee([2020, 2021], [0.05, 0.02])}. Une "
    "initialisation a 0 revient a ajouter en secret une annee a 0 % dans les donnees.",
)

Le piège, démontré côte à côte. La version fausse ci-dessous ne diffère que par sa première ligne —
`pire = 0.0` au lieu de `pire = perfs[0]`. Sur les dix-sept années du portefeuille elle donne la
**bonne** réponse, ce qui la rend indétectable ; sur une série qui ne perd jamais, elle rend une
année qui n'a jamais existé, sans le moindre message.

In [ ]:
"""The two versions side by side: identical on the red thread, opposite on a rising series."""


def pire_annee_zero(annees: list, perfs: list) -> tuple:
    """Wrong on purpose: the accumulator starts at 0 instead of at an observed value."""
    pire = 0.0
    annee_pire = annees[0]
    for annee, perf in zip(annees, perfs):
        if perf < pire:
            pire = perf
            annee_pire = annee
    return annee_pire, pire


hausses_annees, hausses_perfs = [2020, 2021], [0.05, 0.02]
print(f"sur les 17 annees du fil rouge : juste {pire_annee(ANNEES, PERFS_PTF)} | "
      f"faux {pire_annee_zero(ANNEES, PERFS_PTF)}")
print(f"sur deux annees qui montent    : juste {pire_annee(hausses_annees, hausses_perfs)} | "
      f"faux {pire_annee_zero(hausses_annees, hausses_perfs)}")

verifier(
    pire_annee_zero(ANNEES, PERFS_PTF) == pire_annee(ANNEES, PERFS_PTF)
    and pire_annee_zero(hausses_annees, hausses_perfs) != (2021, 0.02),
    "OK : les deux versions sont indiscernables sur les donnees du cours, et opposees des "
    "qu'aucune annee n'est negative -- la fausse rend +0,00 %, une annee qui n'existe pas. "
    "C'est la definition d'un bug muet.",
    "la demonstration n'a pas eu lieu. Cette cellule compare VOTRE pire_annee a la version "
    "fausse : tant que la cellule precedente n'est pas au vert, ce message ne dit rien de "
    "plus qu'elle -- corrigez d'abord pire_annee, puis relancez celle-ci. Si la cellule "
    "precedente est au vert et que celle-ci echoue, alors seulement regardez "
    "pire_annee_zero, qui est fournie et ne doit pas etre modifiee.",
)

In [ ]:
"""Figure -- the seventeen calendar years, worst year annotated (provided)."""
annee_ptf, perf_ptf = pire_annee(ANNEES, PERFS_PTF)

fig, ax = plt.subplots(figsize=(8.0, 4.2))
couleurs = []
for p in PERFS_PTF:
    couleurs.append("#c0392b" if p < 0 else "#2e5f8a")
ax.bar(ANNEES, [100 * p for p in PERFS_PTF], color=couleurs, width=0.68)
ax.axhline(0.0, color="black", lw=0.9)
ax.annotate(f"{annee_ptf} : {perf_ptf:+.2%}", xy=(annee_ptf, 100 * perf_ptf),
            xytext=(10, -24), textcoords="offset points", fontsize=9.5, color="#c0392b",
            arrowprops=dict(arrowstyle="->", color="#c0392b", lw=0.9))
ax.set_title("Dix-sept annees du portefeuille : deux seulement sont negatives")
ax.set_xlabel("annee civile")
ax.set_ylabel("performance de l'annee (%)")
ax.set_xticks(ANNEES[::2])
ax.margins(y=0.20)
plt.show()

negatives = 0
for perf in PERFS_PTF:
    if perf < 0.0:
        negatives += 1
verifier(
    negatives == 2,
    f"OK : {negatives} annees negatives sur 17 -- 2018 a -4,11 % et 2022 a -15,28 %.",
    f"attendu 2 annees negatives ; obtenu {negatives}.",
)

In [ ]:
"""Aide de la partie -- retirez le # d'une ligne APRES avoir essaye."""
# indice(...) donne une etape du raisonnement ; solution(...) donne la reponse.
# indice("Trois lignes avant la boucle : pire = perfs[0] et annee_pire = annees[0]. Dans la boucle, "
#        "un seul if, et DEUX affectations a l'interieur de ce if. Si une seule des deux y est, "
#        "vous afficherez la performance d'une annee avec l'etiquette d'une autre.")
# solution("pire = perfs[0] ; annee_pire = annees[0] ; for annee, perf in zip(annees, perfs): "
#          "if perf < pire: pire = perf ; annee_pire = annee ; return annee_pire, pire")

---

## Partie E — Une fonction nomme un calcul, un `assert` le met à l'épreuve sur un cas connu (15 min, u05)

**Le problème.** Vous venez de calculer un rendement (partie B) et une pire année (partie D). Le
cours va les refaire sans arrêt : rien que la variation quotidienne des six lignes sur 4 151
séances, cela fait 24 906 calculs identiques. Recopier la ligne à chaque fois échoue pour trois
raisons — on la retape mal une fois sur vingt, on ne peut pas la tester isolément, et le jour où
l'on trouve une erreur dedans il faut retrouver ses vingt copies.

Deux fonctions à écrire. La première, `rendement`, est la partie B rendue réutilisable. La seconde
est nouvelle et vaut d'être prédite avant d'être codée.

> **Le gain à l'échéance d'un droit d'achat.** Un contrat vous donne, dans un an, le **droit** — pas
> l'obligation — d'acheter une unité de S&P 500 au prix fixé aujourd'hui de $K = 7\,718{,}60$.
> Notons $S$ le prix de l'indice le jour où ce droit s'éteint. Ce droit rapporte alors
>
> $$\text{payoff}_{\text{call}}(S, K) = \max(S - K,\ 0)$$
>
> soit la différence si elle est positive, et zéro sinon : un droit qu'on n'exerce pas ne coûte
> rien à l'échéance. Ce qu'un tel contrat vaut *avant* l'échéance et pourquoi on en achète, c'est
> le sujet de **ch01 u05** ; ici, ce n'est qu'un calcul à programmer.

**Prédisez avant de coder** : si l'indice vaut 8 500, ce droit rapporte … ? Et s'il vaut 7 000 ?
Les deux réponses sont **781,40** et **0** — pas $7\,000 - 7\,718{,}60 = -718{,}60$. Ces deux
nombres sont exactement les deux `verifier(...)` qui suivent : un cas de contrôle n'est pas un
supplément de rigueur, c'est la réponse que vous aviez déjà en tête avant de coder.

In [ ]:
def rendement(prix_debut: float, prix_fin: float) -> float:
    """Return the simple return between a starting price and an ending price.

    Parameters
    ----------
    prix_debut : float
        Price at the beginning of the period, strictly positive.
    prix_fin : float
        Price at the end of the period.

    Returns
    -------
    float
        ``prix_fin / prix_debut - 1``, in decimal: 0.05 means +5 %.
    """
    # TODO: Renvoyer le rendement simple decrit par la docstring, en decimal.
    raise NotImplementedError("TODO")


def mon_payoff_call(S: float, K: float) -> float:
    """Return the terminal gain of the right to buy at K an asset worth S.

    Parameters
    ----------
    S : float
        Price of the asset on the day the right expires.
    K : float
        Strike: the price written in the contract.

    Returns
    -------
    float
        ``max(S - K, 0.0)``.  Never negative: a right one does not exercise costs
        nothing at maturity.
    """
    # TODO: Renvoyer max(S - K, 0.0) : c'est le max qui ecrit « je n'exerce pas » ;
    #      sans lui, la fonction renvoie un gain negatif.
    raise NotImplementedError("TODO")

In [ ]:
"""Check both functions, including against the payoff_call provided by the course module."""
verifier(
    abs(rendement(7747.71, 7718.60) + 0.0037572392) < 1e-9,
    f"OK : rendement(7747.71, 7718.60) = {rendement(7747.71, 7718.60):.4%}, la meme "
    "baisse qu'en partie B, calculee sur les deux clotures arrondies au centieme.",
    "attendu -0,0037572392 : la formule est prix_fin / prix_debut - 1, dans cet ordre.",
)
verifier(
    abs(rendement(100.0, 110.0) - 0.10) < 1e-12 and abs(rendement(110.0, 100.0) + 0.0909090909) < 1e-9,
    "OK : 100 -> 110 rend +10,00 %, et 110 -> 100 rend -9,09 %. Les deux ne se compensent "
    "pas, alors que le prix est revenu a son point de depart : c'est le point de depart du "
    "ch01 u01.",
    "attendu +0,10 puis -0,0909090909.",
)

verifier(
    abs(mon_payoff_call(8500.0, K) - 781.40) < 1e-9,
    f"OK : a 8 500, le droit d'acheter a {K:,.2f} rapporte "
    f"{mon_payoff_call(8500.0, K):.2f}.",
    "attendu 781,40 : la difference 8500 - 7718,60.",
)
verifier(
    mon_payoff_call(7000.0, K) == 0.0 and mon_payoff_call(K, K) == 0.0,
    "OK : a 7 000 comme a 7 718,60 exactement, le gain est nul -- le droit ne s'exerce pas, "
    "et il ne rapporte rien quand le prix est pile au niveau du contrat.",
    f"attendu 0.0 dans les deux cas ; obtenu {mon_payoff_call(7000.0, K)} et "
    f"{mon_payoff_call(K, K)}. Si vous lisez -718,60, il manque le max (u05).",
)
verifier(
    isinstance(mon_payoff_call(7000.0, K), float),
    "OK : la fonction rend bien un flottant, y compris quand le gain est nul.",
    f"attendu un float ; obtenu un {type(mon_payoff_call(7000.0, K)).__name__}. "
    "Vous avez ecrit max(S - K, 0) avec un zero ENTIER : quand le droit ne s'exerce pas, "
    "la fonction rend l'entier 0 et non 0.0, et un tableau construit a partir d'elle "
    "changera de type sans prevenir. Ecrivez max(S - K, 0.0).",
)

CAS = [5000.0, 7000.0, 7718.60, 8500.0, 11000.0]
ecart_max = 0.0
for S in CAS:
    ecart = abs(mon_payoff_call(S, K) - float(payoff_call(S, K)))
    if ecart > ecart_max:
        ecart_max = ecart
verifier(
    ecart_max < 1e-12,
    f"OK : sur les cinq cas de controle, votre fonction et le payoff_call du module "
    f"different d'au plus {ecart_max:.1e}.",
    f"ecart maximal {ecart_max:.3e} avec payoff_call du module : comparez vos deux sorties "
    "cas par cas, en commencant par S = 5000 (les deux doivent rendre 0.0).",
)

Le piège de l'unité : « ma fonction marche, je l'ai regardée ». Voici la version que presque tout le
monde écrit d'abord — `return S - K`, sans le `max`. Elle donne la bonne réponse sur le cas qu'on
regarde, et un gain **négatif** sur un droit qu'on n'est pas obligé d'exercer. Relue à l'œil, elle
passe. Un `assert` l'arrête en trois secondes.

In [ ]:
"""The u05 trap: a plausible function that is wrong, and the one line that catches it."""


def payoff_call_faux(S: float, K: float) -> float:
    """Wrong on purpose: the max is missing, so an unexercised right shows a loss."""
    return S - K


print(f"a 8 500 : juste {mon_payoff_call(8500.0, K):.2f} | faux {payoff_call_faux(8500.0, K):.2f}")
print(f"a 7 000 : juste {mon_payoff_call(7000.0, K):.2f} | faux {payoff_call_faux(7000.0, K):.2f}")

attrape = False
try:
    assert payoff_call_faux(7000.0, K) == 0.0
except AssertionError:
    attrape = True
    print("AssertionError : la fonction rend -718,60 au lieu de 0,0")

verifier(
    attrape and abs(payoff_call_faux(8500.0, K) - mon_payoff_call(8500.0, K)) < 1e-9,
    "OK : les deux versions sont identiques sur le cas qu'on regarde, et l'assert attrape "
    "la fausse sur le cas qu'on ne regarde pas. Un test qui echoue coute trois secondes, "
    "un prix faux coute un chapitre.",
    "l'assert n'a pas echoue : payoff_call_faux devrait rendre -718,60 pour S = 7 000.",
)

In [ ]:
"""Figure -- the payoff, evaluated by a Python loop over 200 prices (provided)."""
prix_essai = np.linspace(5000.0, 11000.0, 200)
gains = []
for S in prix_essai:
    gains.append(mon_payoff_call(float(S), K))

fig, ax = plt.subplots()
ax.plot(prix_essai, gains, color="#2e5f8a")
ax.plot([7000.0, 8500.0], [mon_payoff_call(7000.0, K), mon_payoff_call(8500.0, K)],
        "o", color="#c0392b", label="vos deux predictions : 0 et 781,40")
ax.axvline(K, color="0.6", lw=0.8, ls="--")
ax.set_title(f"Gain a l'echeance du droit d'acheter a K = {K:,.2f}")
ax.set_xlabel("prix de l'indice le jour de l'echeance")
ax.set_ylabel("gain a l'echeance")
ax.legend()
plt.show()

verifier(
    min(gains) == 0.0 and abs(max(gains) - (11000.0 - K)) < 1e-9,
    f"OK : la courbe est plate a 0 sous {K:,.2f} puis monte a 45 degres ; son minimum est "
    f"{min(gains):.1f} et son maximum {max(gains):.2f}.",
    "attendu un minimum de 0,0 : un gain a l'echeance ne descend jamais sous zero.",
)

In [ ]:
"""Aide de la partie -- retirez le # d'une ligne APRES avoir essaye."""
# indice(...) donne une etape du raisonnement ; solution(...) donne la reponse.
# indice("max(a, b) rend le plus grand des deux nombres. Ecrivez-le avec 0.0, pas avec 0 : le "
#        "cours travaille en flottants, et payoff_call(7000, K) doit rendre 0.0, pas 0.")
# solution("return prix_fin / prix_debut - 1.0 pour rendement, et return max(S - K, 0.0) pour "
#          "mon_payoff_call. C'est le max, et lui seul, qui encode 'je n'exerce pas'.")

---

## Partie F — 10 000 valeurs pour le prix d'une (20 min, u06)

**Le problème.** Une question de gestion des risques, posée telle quelle sur un desk : **combien de
séances ont perdu plus de 2 % en une seule journée depuis 2010 ?** Avec la boucle de la partie D, la
réponse s'écrit en cinq lignes qui tournent 4 151 fois. Le problème est ailleurs : à partir du
chapitre 4, le cours manipulera des tableaux de cent mille à dix millions de nombres, où la même
boucle prendrait des minutes. Il faut une écriture qui vaille aussi bien pour 4 151 valeurs que
pour 10 000 000.

Quatre fonctions, une ligne chacune, et un chronomètre.

### Avant tout : trois prédictions écrites

Cette partie tire deux fois des nombres au sort — non pas pour simuler quoi que ce soit (aucune
simulation dans ce chapitre), mais pour montrer qu'une graine fixée redonne exactement la même
suite. La règle du cours vaut quand même : **on écrit sa prédiction avant, jamais après**.

1. **`rapport_boucle_vecteur`** — sur un million de valeurs, combien de fois la version en tableau
   est-elle plus rapide que la boucle Python ? (un nombre, pas un intervalle)
2. **`memes_nombres`** — `np.random.default_rng(42).standard_normal(3)`, exécuté deux fois de
   suite : les trois nombres sont-ils identiques ? (`True` ou `False`)
3. **`ecart_ddof_pct`** — sur 20 valeurs, `x.std()` et `x.std(ddof=1)` diffèrent de combien, **en
   pour-cent** ?

In [ ]:
def predictions_partie_F() -> dict:
    """The three numbers written down BEFORE part F runs.

    Returns
    -------
    dict
        Keys ``rapport_boucle_vecteur`` (how many times faster the array version is),
        ``memes_nombres`` (bool: does a fixed seed give back the same three numbers?)
        and ``ecart_ddof_pct`` (relative gap between std(ddof=0) and std(ddof=1) on 20
        values, in percent).
    """
    # TODO: Renvoyer vos trois predictions dans un dictionnaire portant exactement
    #      les trois cles de la docstring ; rien n'est note ici.
    raise NotImplementedError("TODO")


PREDICTIONS_F = predictions_partie_F()

assert set(PREDICTIONS_F) == {
    "rapport_boucle_vecteur",
    "memes_nombres",
    "ecart_ddof_pct",
}, "the dict must carry exactly the three keys of the docstring"
assert PREDICTIONS_F["rapport_boucle_vecteur"] > 0.0, (
    "rapport_boucle_vecteur attend un nombre strictement positif (combien de fois plus vite)"
)
assert isinstance(PREDICTIONS_F["memes_nombres"], bool), (
    "memes_nombres attend True ou False, pas un texte : ecrivez True, sans guillemets"
)
for cle, valeur in PREDICTIONS_F.items():
    print(f"{cle:>24s} : {valeur}")

### Les quatre fonctions

| Fonction | Ce qu'elle fait | La primitive |
|---|---|---|
| `compter_seances_sous(rendements, seuil)` | compte les séances sous un seuil, **sans boucle** | masque booléen, puis `.sum()` |
| `payoff_call_vec(S, K)` | le payoff de la partie E, sur tout un tableau | `np.maximum` |
| `somme_des_gains(prix, K)` | la somme de ces payoffs, en **une** expression | `np.maximum(...).sum()` |
| `modifier_par_tranche(a)` | écrit 40,0 dans la tranche `a[3:]` et rend `a` | tranche = **vue** |

La dernière n'est pas un exercice de style : elle fabrique exprès le piège le plus coûteux du
calcul numérique. Sur une **liste**, une tranche produit une nouvelle liste ; sur un **tableau**,
elle produit une **vue** — un second nom regardant la même mémoire.

In [ ]:
def compter_seances_sous(rendements: np.ndarray, seuil: float) -> int:
    """Count the sessions whose return is strictly below `seuil`.

    Parameters
    ----------
    rendements : ndarray, shape (n,)
        Daily returns, in decimal.
    seuil : float
        Threshold, in decimal: ``-0.02`` for -2 %.

    Returns
    -------
    int
        How many cells of `rendements` are strictly below `seuil`.  No Python loop:
        the comparison must be written once, for the whole array.
    """
    # TODO: Comparer le tableau entier a seuil, ce qui donne un masque de booleens,
    #      sommer ce masque (True compte pour 1) et renvoyer le resultat converti en
    #      int.
    raise NotImplementedError("TODO")


def payoff_call_vec(S, K: float):
    """Vectorised payoff of a call: S may be a scalar or an array of any shape.

    Parameters
    ----------
    S : float or ndarray
        Price(s) of the asset at maturity.
    K : float
        Strike.

    Returns
    -------
    float or ndarray
        ``max(S - K, 0)`` applied cell by cell, same shape as `S`.
    """
    # TODO: Renvoyer np.maximum(S - K, 0.0) : la version « case par case » du max de
    #      la partie E.
    raise NotImplementedError("TODO")


def somme_des_gains(prix: np.ndarray, K: float) -> float:
    """Return the sum of the call payoffs over a whole array of prices.

    Parameters
    ----------
    prix : ndarray, shape (n,)
        Prices at maturity.
    K : float
        Strike.

    Returns
    -------
    float
        The sum of ``max(prix - K, 0)``, written as a single numpy expression: no
        intermediate Python list, no loop.
    """
    # TODO: Renvoyer float(np.maximum(prix - K, 0.0).sum()) : une seule expression,
    #      appliquee au tableau entier d'un coup.
    raise NotImplementedError("TODO")


def modifier_par_tranche(a: np.ndarray) -> np.ndarray:
    """Write 40.0 through the slice ``a[3:]`` and return the ORIGINAL array.

    Parameters
    ----------
    a : ndarray, shape (n,)
        Array with at least four cells.

    Returns
    -------
    ndarray
        The very same array `a`, after writing 40.0 in the first cell of the slice.
        Nothing is copied here: the point of the exercise is to see what a slice shares
        with the array it comes from.
    """
    # TODO: Lier b a la tranche a[3:], mettre b[0] a 40.0, puis renvoyer a ; ne rien
    #      copier.
    raise NotImplementedError("TODO")

In [ ]:
"""Answer the risk question of the part, and check the vectorised payoff."""
RENDEMENTS_SP = RLOG["SP500"]
SEANCES_SOUS_2 = compter_seances_sous(RENDEMENTS_SP, -0.02)
pire_seance = float(RENDEMENTS_SP.min())
date_pire = DATES[int(RENDEMENTS_SP.argmin())]
ecart_type_annualise = float(RENDEMENTS_SP.std(ddof=1) * np.sqrt(252))

# The panel holds as many daily variations as sessions: the first one is computed
# against the 2010-01-04 close, the day before the panel starts (u06).
print(f"forme du tableau        : {RENDEMENTS_SP.shape}")
print(f"seances sous -2 %       : {SEANCES_SOUS_2} sur {RENDEMENTS_SP.size} "
      f"({100 * SEANCES_SOUS_2 / RENDEMENTS_SP.size:.2f} %)")
print(f"pire seance             : {pire_seance:.4f} le {date_pire}")
print(f"ecart-type x sqrt(252)  : {ecart_type_annualise:.4f}")

verifier(
    SEANCES_SOUS_2 == 139,
    f"OK : {SEANCES_SOUS_2} seances sur 4 151 ont perdu plus de 2 % en un jour, soit une "
    "toutes les six semaines environ.",
    f"attendu 139 ; obtenu {SEANCES_SOUS_2}. Le masque est (rendements < seuil), et sa "
    "somme compte les True (True vaut 1, False vaut 0). Un resultat non entier, de l'ordre "
    "de 1,9, signifie que vous avez somme les rendements au lieu du masque ; un compte de "
    "4151 signifie que vous avez rendu la TAILLE du tableau, len(rendements) ou .size ; un "
    "compte de 138 signifie que vous avez recalcule les variations depuis les prix avec la "
    "formule d'u02 -- ici on prend la colonne du fichier telle qu'elle est fournie (u06).",
)
verifier(
    abs(pire_seance + 0.1277) < 1e-4 and str(date_pire) == "2020-03-16",
    f"OK : la pire seance est le {date_pire}, a {pire_seance:.4f} sur l'echelle du "
    "fichier, soit une baisse de 11,98 % -- pres du huitieme de la valeur de l'indice en "
    "une seule journee. Ne comparez pas ce nombre a la pire ANNEE du portefeuille "
    "(-15,28 %) : ni le meme actif, ni la meme duree, ni la meme facon de compter.",
    "attendu -0,1277 le 2020-03-16, sur l'echelle du fichier.",
)
verifier(
    abs(ecart_type_annualise - 0.1731) < 1e-4,
    f"OK : l'ecart-type des 4 151 valeurs, multiplie par racine de 252, vaut "
    f"{ecart_type_annualise:.4f}. Ce que ce nombre mesure, et pourquoi cette racine, "
    "c'est le ch01 u02 : ici, c'est une operation numerique sur un tableau, rien de plus.",
    f"attendu 0,1731 ; obtenu {ecart_type_annualise:.4f}. Avec ddof=0 (le defaut de numpy) "
    "vous obtiendriez 0,17305 : ecrivez ddof=1, la convention du cours.",
)

prix_grille = np.linspace(5000.0, 11000.0, 10000)
gains_vec = payoff_call_vec(prix_grille, K)
ecart_vec = float(np.abs(gains_vec - payoff_call(prix_grille, K)).max())
verifier(
    gains_vec.shape == (10000,) and ecart_vec < 1e-12,
    f"OK : 10 000 gains calcules en une ligne, identiques au payoff_call du module "
    f"(ecart maximal {ecart_vec:.1e}). La figure d'u05 demandait une boucle sur 200 prix.",
    f"attendu un tableau de forme (10000,) egal au module a 1e-12 ; ecart {ecart_vec:.3e}. "
    "np.maximum compare case par case ; max, lui, refuse un tableau.",
)

In [ ]:
"""The silent trap of u06: a slice of an array is a view, a slice of a list is a copy."""
a = np.array([10.0, 2.0, 3.0, 4.0, 5.0, 6.0])
apres_tableau = modifier_par_tranche(a)

liste = [10.0, 2.0, 3.0, 4.0, 5.0, 6.0]
morceau = liste[3:]
morceau[0] = 40.0

print(f"tableau apres ecriture dans a[3:] : {apres_tableau}")
print(f"liste   apres ecriture dans l[3:] : {liste}")

verifier(
    np.allclose(apres_tableau, [10.0, 2.0, 3.0, 40.0, 5.0, 6.0]),
    "OK : ecrire dans la tranche a modifie le tableau d'origine -- une tranche de tableau "
    "est une VUE, un second nom sur la meme memoire.",
    f"attendu [10. 2. 3. 40. 5. 6.] ; obtenu {apres_tableau}. Si rien n'a bouge, vous avez "
    "copie la tranche (a[3:].copy()) : ici, c'est justement la vue qu'on veut observer.",
)
verifier(
    liste == [10.0, 2.0, 3.0, 4.0, 5.0, 6.0],
    "OK : la meme operation sur une liste ne touche a rien. Meme syntaxe, deux "
    "comportements : c'est ce qui rend ce piege muet.",
    f"la liste a bouge : {liste}.",
)

masque = a > 3.0
copie_par_masque = a[masque]
copie_par_masque[0] = 0.0
verifier(
    a[0] == 10.0,
    "OK : un masque booleen, lui, fabrique une copie -- a[a > 3] se modifie sans danger, "
    "a[3:] non. La parade dans les deux cas s'ecrit a[3:].copy().",
    f"attendu a[0] = 10.0 ; obtenu {a[0]}.",
)

In [ ]:
"""Measure the gain of vectorisation, then draw the two panels of the part."""
tableau = np.linspace(5000.0, 11000.0, 1_000_000)
liste_valeurs = tableau.tolist()

debut = time.perf_counter()
total_boucle = 0.0
for x in liste_valeurs:
    total_boucle += max(x - K, 0.0)
DUREE_BOUCLE = time.perf_counter() - debut

debut = time.perf_counter()
TOTAL_VECTEUR = somme_des_gains(tableau, K)
DUREE_VECTEUR = time.perf_counter() - debut
RAPPORT = DUREE_BOUCLE / DUREE_VECTEUR

print(f"boucle Python : {DUREE_BOUCLE:.4f} s | tableau numpy : {DUREE_VECTEUR:.5f} s "
      f"| rapport {RAPPORT:.0f}")

verifier(
    abs(total_boucle - TOTAL_VECTEUR) < 1e-6 * abs(TOTAL_VECTEUR),
    f"OK : les deux methodes rendent le meme total ({TOTAL_VECTEUR:,.2f}). Le gain de "
    "temps ne se paie donc pas en justesse.",
    f"les deux totaux different : {total_boucle} contre {TOTAL_VECTEUR}. La version "
    "vectorisee doit calculer exactement max(prix - K, 0) sur les memes valeurs.",
)
verifier(
    RAPPORT > 5.0,
    f"OK : {RAPPORT:.0f} fois plus rapide sur un million de valeurs. Le facteur depend de "
    "votre machine -- on le MESURE, on ne le recite pas -- et il tient a deux causes : "
    "une verification de type a chaque tour de boucle, et une memoire contigue contre une "
    "liste d'adresses.",
    f"rapport mesure {RAPPORT:.2f}, attendu au moins 5. Si somme_des_gains contient une "
    "boucle Python -- un for, sous quelque forme que ce soit -- elle n'est pas vectorisee : "
    "l'operation doit s'ecrire une seule fois, sur le tableau entier.",
)

fig, (ax_g, ax_d) = plt.subplots(1, 2, figsize=(10.0, 4.0))
ax_g.hist(100 * RENDEMENTS_SP, bins=80, color="#2e5f8a")
ax_g.axvspan(100 * RENDEMENTS_SP.min() - 0.5, -2.0, color="#c0392b", alpha=0.18)
ax_g.set_title(f"4 151 variations quotidiennes\n{SEANCES_SOUS_2} seances sous -2 %")
ax_g.set_xlabel("variation quotidienne (%)")
ax_g.set_ylabel("nombre de seances")
ax_d.bar(["boucle", "tableau"], [DUREE_BOUCLE, DUREE_VECTEUR], color=["#c0392b", "#2e5f8a"])
ax_d.set_yscale("log")
ax_d.set_title(f"Un million de valeurs : rapport {RAPPORT:.0f}")
ax_d.set_ylabel("temps mesure (s, echelle log)")
for i, duree in enumerate([DUREE_BOUCLE, DUREE_VECTEUR]):
    ax_d.annotate(f"{duree:.4f} s", xy=(i, duree), xytext=(0, 6),
                  textcoords="offset points", ha="center", fontsize=9)
plt.show()

### L'encadré des graines, et celui de `ddof`

Deux cellules fournies, deux fautes muettes de moins. La première montre qu'une graine fixée
redonne la même suite, et que la seule façon correcte de dériver un second flux est `rng.spawn(k)` —
la documentation de NumPy écrit « UNSAFE! Do not do this! » à propos de `SEED + i`, dont l'erreur
ne lève aucun message.

La seconde compare `std()` et `std(ddof=1)`. L'écart n'est pas une question d'échantillon : il vaut
exactement $\sqrt{N/(N-1)} - 1$, soit **2,5978 %** sur 20 valeurs et 0,00005 % sur un million.
Invisible sur les grands échantillons, il fausse tous les petits — donc on écrit `ddof=1`, partout.

In [ ]:
"""One SEED, one rng, and spawn: the reproducibility box of u06 (provided)."""
premier = np.random.default_rng(SEED).standard_normal(3)
second = np.random.default_rng(SEED).standard_normal(3)
enfants = rng.spawn(2)
flux_a = enfants[0].standard_normal(3)
flux_b = enfants[1].standard_normal(3)

print(f"default_rng({SEED}), premier appel : {np.round(premier, 6)}")
print(f"default_rng({SEED}), second appel  : {np.round(second, 6)}")
print(f"rng.spawn(2), enfant 0            : {np.round(flux_a, 6)}")
print(f"rng.spawn(2), enfant 1            : {np.round(flux_b, 6)}")

MEMES_NOMBRES = bool(np.array_equal(premier, second))
verifier(
    MEMES_NOMBRES,
    "OK : meme graine, meme suite, sur n'importe quelle machine. C'est ce qui rend un "
    "resultat verifiable par quelqu'un d'autre.",
    "les deux appels devraient rendre les trois memes nombres.",
)
verifier(
    not np.array_equal(flux_a, flux_b),
    "OK : les deux enfants de spawn produisent des flux differents, et leur independance "
    "est construite -- alors que default_rng(SEED + i) ne garantit rien du tout.",
    "les deux enfants rendent la meme suite : verifiez rng.spawn(2).",
)
predictions_f = globals().get("PREDICTIONS_F", {})
print(f"votre prediction 'memes_nombres' : "
      f"{predictions_f.get('memes_nombres', '(a completer)')} | mesure : {MEMES_NOMBRES}")

In [ ]:
"""ddof: the same dispersion, two conventions, one exact gap (provided)."""
echantillon = rng.spawn(1)[0].standard_normal(20)
sans_ddof = float(echantillon.std())
avec_ddof = float(echantillon.std(ddof=1))
ECART_DDOF_PCT = 100.0 * (avec_ddof / sans_ddof - 1.0)

print(f"std()        (ddof=0, defaut numpy) : {sans_ddof:.7f}")
print(f"std(ddof=1)  (convention du cours)  : {avec_ddof:.7f}")
print(f"ecart : {ECART_DDOF_PCT:.4f} % sur 20 valeurs")

verifier(
    abs(ECART_DDOF_PCT - 2.5978) < 1e-3,
    f"OK : {ECART_DDOF_PCT:.4f} % d'ecart, et ce nombre ne depend pas du tirage : il vaut "
    "racine de N/(N-1) moins 1. Sur un million de valeurs il tombe a 0,00005 %, donc "
    "invisible, donc jamais detecte.",
    f"attendu 2,5978 % ; obtenu {ECART_DDOF_PCT:.4f} %.",
)
predictions_f = globals().get("PREDICTIONS_F", {})
print(f"votre prediction 'ecart_ddof_pct' : "
      f"{predictions_f.get('ecart_ddof_pct', '(a completer)')} % | "
      f"mesure : {ECART_DDOF_PCT:.4f} %")
print(f"votre prediction 'rapport_boucle_vecteur' : "
      f"{predictions_f.get('rapport_boucle_vecteur', '(a completer)')} | "
      f"mesure : {rappel('RAPPORT', '{:.0f}')}")

In [ ]:
"""Aide de la partie -- retirez le # d'une ligne APRES avoir essaye."""
# indice(...) donne une etape du raisonnement ; solution(...) donne la reponse.
# indice("Les quatre fonctions font une ligne chacune. Le masque booleen s'ecrit "
#        "(rendements < seuil) et se compte par .sum() ; np.maximum(S - K, 0.0) remplace le max ; "
#        "et pour la vue, il suffit de NE PAS copier : b = a[3:] puis b[0] = 40.0.")
# solution("return int((rendements < seuil).sum()) ; return np.maximum(S - K, 0.0) ; "
#          "return float(np.maximum(prix - K, 0.0).sum()) ; b = a[3:] ; b[0] = 40.0 ; return a.")

---

## Partie G — Confrontation (5 min)

Relisez vos trois prédictions de la partie 0 à côté des trois chiffres mesurés, et écrivez
**trois lignes** : pour chacune, l'écart, et ce qui l'explique. Ce n'est pas un exercice de style.
Une prédiction qu'on ne relit pas ne sert à rien ; c'est en nommant l'écart qu'on cesse de le
refaire.

Trois pistes, si vous cherchez quoi écrire :

- combien de minutes vous aviez annoncées, et où vous situiez la barrière d'entrée ;
- si vous aviez nommé 2020 pour la pire année : la courbe d'u01 montre le **chemin**, le tableau
  d'u04 ne retient que **deux dates par année**, et 2020 s'est refermée à +21,16 % ;
- si vous aviez sous-estimé les 139 séances : elles représentent 3,35 % du total, soit environ une
  séance par mois et demi.

In [ ]:
def commentaire() -> str:
    """Three written lines confronting the three predictions with the three measures.

    Returns
    -------
    str
        At least 80 characters: one sentence per prediction, saying the gap and what
        explains it.
    """
    # TODO: Renvoyer vos trois lignes, au moins 80 caracteres, une phrase par
    #      prediction ; le style n'est pas note.
    raise NotImplementedError("TODO")


COMMENTAIRE = commentaire()
print(COMMENTAIRE)
verifier(
    len(COMMENTAIRE) > 80,
    f"OK : {len(COMMENTAIRE)} caracteres ecrits. La confrontation est le seul moment du TP "
    "ou vous produisez du texte, et c'est celui qui laisse une trace.",
    f"attendu plus de 80 caracteres ; obtenu {len(COMMENTAIRE)}. Une phrase par prediction.",
)

In [ ]:
"""Confrontation table: predicted, measured, and the usual explanation of the gap."""
# The lab cannot time the reader; two cells is what the measured path costs, and the
# table says so in clear rather than pretending to have measured it.
mesures = {
    "minutes_jusqu_a_la_figure": 2.0,
    "pire_annee_pct": 100.0 * min(PERFS_PTF),
    "seances_sous_moins_2pct": float(int((RLOG["SP500"] < -0.02).sum())),
}
commentaires = {
    "minutes_jusqu_a_la_figure": "non mesure : deux cellules suffisent (cellule 1, puis tracer_actif)",
    "pire_annee_pct": "2022, et non 2020 (+21,16 %) ; la meilleure est 2019 a +34,68 %",
    "seances_sous_moins_2pct": "3,35 % des 4 151 seances ; la pire, -12,77 % le 2020-03-16",
}
predictions = globals().get("PREDICTIONS", {})

print(f"{'grandeur':<28s} {'predit':>10s} {'mesure':>10s}   commentaire")
for cle, mesure in mesures.items():
    predit = predictions.get(cle)
    texte = f"{float(predit):>10.2f}" if predit is not None else f"{'(a completer)':>10s}"
    print(f"{cle:<28s} {texte} {mesure:>10.2f}   {commentaires[cle]}")

print(f"\nrappel de vos predictions brutes : {rappel('PREDICTIONS')}")
print(f"rapport boucle / tableau mesure  : {rappel('RAPPORT', '{:.0f}')}")
print("Fin des sept parties. Vous savez desormais garder une valeur, repeter un geste et "
      "nommer un calcul -- les trois seules choses que la machine fera de tout le cours.")

---

## Partie H — Passer en local (facultatif ici, **obligatoire avant le ch05**, u07)

À partir du chapitre 5, une seule simulation fabrique 100 000 scénarios de 252 pas, soit 25,2
millions de nombres : Colab n'est plus le bon outil, et le projet final sera exécuté **chez le
correcteur**. L'unité u07 installe l'environnement du cours et, surtout, apprend à **prouver**
qu'il est bon.

**Le livrable** : la sortie complète de `python check_install.py`, **en texte**, jamais une capture
d'écran — on ne peut ni chercher ni copier dans une image, et les trois lignes qui comptent sont
souvent celles qui dépassent du cadre. Le verdict attendu est `INSTALLATION VALIDE` ; un
`[WARN] numba` n'est pas bloquant.

La cellule ci-dessous affiche les **trois diagnostics** qui règlent presque tous les incidents.
Exécutez-la ici : elle décrit la machine sur laquelle ce notebook tourne en ce moment.

In [ ]:
"""The three diagnostics of u07: which Python, which backend, which numpy (provided)."""
import matplotlib

print(f"1. sys.executable          : {sys.executable}")
print(f"2. matplotlib.get_backend(): {matplotlib.get_backend()}")
print(f"3. numpy.__version__       : {np.__version__}")
print()
print("1. Un ModuleNotFoundError dans le notebook alors que le terminal importe tres bien ?")
print("   Les deux sys.executable different : le noyau Jupyter n'est pas dans le bon")
print("   environnement. C'est l'incident numero un du cours.")
print("2. Aucune figure ne s'affiche ? Un backend 'Agg' dessine dans un fichier, pas a")
print("   l'ecran ; dans un notebook, %matplotlib inline est la reponse.")
print("3. La cellule ne finit pas ? Reduire d'un facteur 100 le nombre de valeurs traitees")
print("   ailleurs, puis relancer avec --ExecutePreprocessor.timeout=900.")

verifier(
    np.__version__.split(".")[0].isdigit(),
    f"OK : numpy {np.__version__} repond, et vous savez desormais quel Python l'a charge.",
    "numpy ne repond pas : reprendre u07 depuis conda activate mc-dp-finance.",
)